# 06 LoRA MVP v2 Fast Ablation Colab

本 notebook 用于快速迭代 LoRA MVP v2。v1 已经完成 held-out 评测，但 noise/reverb 变差，因此这里不进入 router，而是做更小、更快的 ablation：默认 attention-only、noise/reverb-only、`lr=1e-5`、`150` steps。训练完成后立即在同一个 MVP 150 held-out test 上评测，并对比 base、v1、v2。


In [ ]:
# 挂载 Google Drive。
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
# 安装最小依赖。
# 当前训练和评测不依赖 torchao；Colab 旧版 torchao 会导致 PEFT 注入/加载 LoRA 失败。
%pip -q install --upgrade --upgrade-strategy only-if-needed qwen-asr==0.0.6 transformers==4.57.6 accelerate==1.12.0 peft==0.19.1 bitsandbytes huggingface_hub pyyaml
%pip -q install pandas==2.2.2 requests==2.32.4
%pip -q uninstall -y torchao


In [ ]:
# 快速实验参数集中在这里。
# 如果要做下一组 ablation，优先只改 CONFIG_PATH、INCLUDE_SCENARIOS、MAX_STEPS 或 LEARNING_RATE。
from pathlib import Path
import json
import yaml

PROJECT_DIR = Path('/content/drive/MyDrive/qwen3-asr')
CONFIG_PATH = PROJECT_DIR / 'configs/train/qwen3_asr_lora_mvp_v2_ablation.yaml'
TRAIN_MANIFEST = PROJECT_DIR / 'data/jsonl/lora_mvp_train.local.jsonl'
HELD_OUT_MANIFEST = PROJECT_DIR / 'data/jsonl/baseline_mvp_150.local.jsonl'
BASE_METRICS = PROJECT_DIR / 'outputs/baseline_mvp_150/metrics.qwen3_asr_base.mvp_150.json'
V1_METRICS = PROJECT_DIR / 'outputs/lora_mvp_eval/metrics.qwen3_asr_lora_mvp.mvp_150.json'

config = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
MODEL_ID = config.get('model', {}).get('id', 'Qwen/Qwen3-ASR-1.7B')
DTYPE = config.get('probe', {}).get('dtype', 'float16')
DEVICE_MAP = config.get('probe', {}).get('device_map', 'cuda:0')
QUANTIZATION = config.get('model', {}).get('quantization', '4bit')
LANGUAGE = 'English'
MAX_NEW_TOKENS = 128
MAX_INFERENCE_BATCH_SIZE = 1

OUTPUT_DIR = PROJECT_DIR / config.get('output', {}).get('checkpoint_dir', 'checkpoints/qwen3-asr-1.7b-lora-mvp-v2-attn-noise-reverb')
EVAL_DIR = PROJECT_DIR / config.get('output', {}).get('eval_dir', 'outputs/lora_mvp_v2_eval')
ADAPTER_DIR = OUTPUT_DIR / 'adapter'
INCLUDE_SCENARIOS = ','.join(config.get('training', {}).get('include_scenarios', ['noise', 'reverb']))
MAX_STEPS = int(config.get('training', {}).get('max_steps', 150))
LEARNING_RATE = float(config.get('training', {}).get('learning_rate', 1e-5))
EXPECTED_TARGET_COUNT = int(config.get('lora', {}).get('expected_target_count', 96))
SAMPLING_STRATEGY = config.get('training', {}).get('sampling_strategy', 'manifest_order')
SAMPLING_BUCKET_FIELDS = ','.join(config.get('training', {}).get('sampling_bucket_fields', []))

PRED_JSONL = EVAL_DIR / 'predictions.qwen3_asr_lora_mvp_v2.mvp_150.jsonl'
SCORED_JSONL = EVAL_DIR / 'predictions.qwen3_asr_lora_mvp_v2.mvp_150.scored.jsonl'
METRICS_JSON = EVAL_DIR / 'metrics.qwen3_asr_lora_mvp_v2.mvp_150.json'
SCENARIO_CSV = EVAL_DIR / 'metrics_by_scenario.qwen3_asr_lora_mvp_v2.mvp_150.csv'
ERROR_DIR = EVAL_DIR / 'error_analysis'
EVAL_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_DIR =', PROJECT_DIR)
print('CONFIG_PATH =', CONFIG_PATH)
print('OUTPUT_DIR =', OUTPUT_DIR)
print('EVAL_DIR =', EVAL_DIR)
print('INCLUDE_SCENARIOS =', INCLUDE_SCENARIOS)
print('MAX_STEPS =', MAX_STEPS, 'LEARNING_RATE =', LEARNING_RATE, 'EXPECTED_TARGET_COUNT =', EXPECTED_TARGET_COUNT)
print('SAMPLING_STRATEGY =', SAMPLING_STRATEGY, 'SAMPLING_BUCKET_FIELDS =', SAMPLING_BUCKET_FIELDS)


In [ ]:
# 检查 GPU 和必要文件。
import subprocess
import torch

print('torch =', torch.__version__)
print('cuda available =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu =', torch.cuda.get_device_name(0))
    print('capability =', torch.cuda.get_device_capability(0))
else:
    raise RuntimeError('当前 runtime 没有 CUDA GPU，无法训练 Qwen3-ASR LoRA。')
subprocess.run(['nvidia-smi'], check=False)

required = [
    CONFIG_PATH,
    TRAIN_MANIFEST,
    HELD_OUT_MANIFEST,
    BASE_METRICS,
    V1_METRICS,
    PROJECT_DIR / 'train/train_qwen3_asr_lora.py',
    PROJECT_DIR / 'inference/qwen3_asr_lora_infer.py',
    PROJECT_DIR / 'evaluation/eval_wer.py',
    PROJECT_DIR / 'evaluation/analyze_errors.py',
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('缺少必要文件:\n' + '\n'.join(missing))


In [ ]:
# 检查训练 manifest，并确认 scenario filter 后覆盖 noise/reverb 的 short/long。
from collections import Counter, defaultdict


def read_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]


def resolve_audio(audio):
    p = Path(audio)
    if p.is_absolute():
        return p
    return PROJECT_DIR / p


def balanced_round_robin(rows, fields):
    buckets = defaultdict(list)
    for row in rows:
        key = tuple(str(row.get(field, '')) for field in fields)
        buckets[key].append(row)
    if len(buckets) <= 1:
        return rows
    ordered = []
    for index in range(max(len(items) for items in buckets.values())):
        for key in sorted(buckets):
            items = buckets[key]
            if index < len(items):
                ordered.append(items[index])
    return ordered

train_rows = read_jsonl(TRAIN_MANIFEST)
scenario_set = {item.strip() for item in INCLUDE_SCENARIOS.split(',') if item.strip()}
bucket_fields = [item.strip() for item in SAMPLING_BUCKET_FIELDS.split(',') if item.strip()]
selected_rows = [row for row in train_rows if row.get('scenario') in scenario_set]
ordered_rows = balanced_round_robin(selected_rows, bucket_fields) if SAMPLING_STRATEGY == 'scenario_bucket_round_robin' else selected_rows
preview_rows = ordered_rows[:MAX_STEPS]

print('train rows =', len(train_rows), 'selected =', len(selected_rows), 'preview_steps =', len(preview_rows))
print('all counts =', dict(Counter(row.get('scenario', '') for row in train_rows)))
print('selected scenario counts =', dict(Counter(row.get('scenario', '') for row in selected_rows)))
print('selected scenario/bucket counts =', dict(Counter((row.get('scenario', ''), row.get('text_length_bucket', '')) for row in selected_rows)))
print('first-step scenario/bucket counts =', dict(Counter((row.get('scenario', ''), row.get('text_length_bucket', '')) for row in preview_rows)))
print('first 12 ordered rows =', [(row.get('scenario'), row.get('text_length_bucket'), row.get('utterance_id')) for row in preview_rows[:12]])

missing = [str(resolve_audio(row['audio'])) for row in selected_rows if not resolve_audio(row['audio']).exists()]
print('missing selected audio =', len(missing))
if missing:
    print('\n'.join(missing[:20]))
    raise FileNotFoundError(f'v2 selected train rows 有缺失音频: {len(missing)}')
if not selected_rows:
    raise ValueError('scenario filter 没有选中任何训练样本')

expected_buckets = {(scenario, bucket) for scenario in scenario_set for bucket in {'short', 'long'}}
covered_buckets = {(row.get('scenario', ''), row.get('text_length_bucket', '')) for row in preview_rows}
missing_buckets = expected_buckets - covered_buckets
if missing_buckets:
    raise RuntimeError(f'{MAX_STEPS} step 预览没有覆盖所有 scenario/bucket: {sorted(missing_buckets)}')


In [ ]:
# 可选：设置 Hugging Face token。
# 如果模型下载遇到权限或限流问题，在 Colab Secrets 里设置 HF_TOKEN 后重跑本 cell。
import os

try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception:
    token = None

if token:
    os.environ['HF_TOKEN'] = token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = token
    print('HF token detected from Colab Secrets.')
else:
    print('No HF token found in Colab Secrets. Public download will be used.')


In [ ]:
# 命令执行工具：打印 stdout/stderr tail，便于快速定位失败点。
import subprocess
import sys


def run_cmd(cmd, stderr_tail=16000, stdout_tail=12000):
    print('运行命令:')
    print(' '.join(map(str, cmd)))
    result = subprocess.run(cmd, cwd=str(PROJECT_DIR), text=True, capture_output=True)
    print('returncode =', result.returncode)
    if result.stdout:
        print('--- stdout tail ---')
        print(result.stdout[-stdout_tail:])
    if result.stderr:
        print('--- stderr tail ---')
        print(result.stderr[-stderr_tail:])
    result.check_returncode()
    return result


In [ ]:
# Preflight：确认 v2 attention-only target 数为 96。
preflight_cmd = [
    sys.executable,
    'train/train_qwen3_asr_lora.py',
    '--config', str(CONFIG_PATH),
    '--manifest', str(TRAIN_MANIFEST),
    '--audio-root', str(PROJECT_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--model-id', MODEL_ID,
    '--dtype', DTYPE,
    '--device-map', DEVICE_MAP,
    '--quantization', QUANTIZATION,
    '--language', LANGUAGE,
    '--include-scenarios', INCLUDE_SCENARIOS,
    '--sampling-strategy', SAMPLING_STRATEGY,
    '--sampling-bucket-fields', SAMPLING_BUCKET_FIELDS,
    '--learning-rate', str(LEARNING_RATE),
    '--max-steps', str(MAX_STEPS),
    '--preflight-only',
]
run_cmd(preflight_cmd)

summary = json.loads((OUTPUT_DIR / 'summary.json').read_text(encoding='utf-8'))
print(json.dumps(summary, ensure_ascii=False, indent=2)[:5000])
assert summary.get('status') == 'preflight_ok', summary
assert summary.get('count') == EXPECTED_TARGET_COUNT, summary.get('count')
print('preflight 验收通过。')


In [ ]:
# v2 快速训练：默认 150 step。
train_cmd = [
    sys.executable,
    'train/train_qwen3_asr_lora.py',
    '--config', str(CONFIG_PATH),
    '--manifest', str(TRAIN_MANIFEST),
    '--audio-root', str(PROJECT_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--model-id', MODEL_ID,
    '--dtype', DTYPE,
    '--device-map', DEVICE_MAP,
    '--quantization', QUANTIZATION,
    '--language', LANGUAGE,
    '--include-scenarios', INCLUDE_SCENARIOS,
    '--sampling-strategy', SAMPLING_STRATEGY,
    '--sampling-bucket-fields', SAMPLING_BUCKET_FIELDS,
    '--learning-rate', str(LEARNING_RATE),
    '--max-steps', str(MAX_STEPS),
]
run_cmd(train_cmd, stderr_tail=18000, stdout_tail=14000)

summary = json.loads((OUTPUT_DIR / 'summary.json').read_text(encoding='utf-8'))
loss_lines = [line for line in (OUTPUT_DIR / 'loss_log.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
print(json.dumps(summary, ensure_ascii=False, indent=2)[:5000])
print('loss lines =', len(loss_lines))
print('last losses:')
print('\n'.join(loss_lines[-10:]))
loss_rows = [json.loads(line) for line in loss_lines]
loss_bucket_counts = Counter((row.get('scenario', ''), row.get('text_length_bucket', '')) for row in loss_rows)
print('loss scenario/bucket counts =', dict(loss_bucket_counts))
expected_buckets = {(scenario, bucket) for scenario in scenario_set for bucket in {'short', 'long'}}
missing_loss_buckets = expected_buckets - set(loss_bucket_counts)
if missing_loss_buckets:
    raise RuntimeError(f'loss log 没有覆盖所有 scenario/bucket: {sorted(missing_loss_buckets)}')
assert summary.get('status') == 'trained', summary
assert summary.get('steps') == MAX_STEPS, summary.get('steps')
assert ADAPTER_DIR.exists(), f'缺少 adapter dir: {ADAPTER_DIR}'
assert len(loss_lines) == MAX_STEPS, len(loss_lines)


In [ ]:
# v2 held-out MVP 150 推理。
infer_cmd = [
    sys.executable,
    'inference/qwen3_asr_lora_infer.py',
    '--manifest', str(HELD_OUT_MANIFEST),
    '--output-jsonl', str(PRED_JSONL),
    '--adapter-dir', str(ADAPTER_DIR),
    '--audio-root', str(PROJECT_DIR),
    '--model-id', MODEL_ID,
    '--dtype', DTYPE,
    '--device-map', DEVICE_MAP,
    '--quantization', QUANTIZATION,
    '--max-inference-batch-size', str(MAX_INFERENCE_BATCH_SIZE),
    '--max-new-tokens', str(MAX_NEW_TOKENS),
    '--language', LANGUAGE,
]
run_cmd(infer_cmd, stderr_tail=18000, stdout_tail=14000)


In [ ]:
# v2 WER/CER 和错误分析。
eval_cmd = [
    sys.executable,
    'evaluation/eval_wer.py',
    '--predictions-jsonl', str(PRED_JSONL),
    '--scored-jsonl', str(SCORED_JSONL),
    '--metrics-json', str(METRICS_JSON),
    '--metrics-by-scenario-csv', str(SCENARIO_CSV),
]
run_cmd(eval_cmd)

analysis_cmd = [
    sys.executable,
    'evaluation/analyze_errors.py',
    '--scored-jsonl', str(SCORED_JSONL),
    '--output-dir', str(ERROR_DIR),
]
run_cmd(analysis_cmd)


In [ ]:
# 对比 base、v1、v2。v2 只要 noise 或 reverb 有一个低于 base，才算进入下一步候选。
import pandas as pd


def load_metrics(path):
    data = json.loads(Path(path).read_text(encoding='utf-8'))
    out = {row['group']: row for row in data.get('by_scenario', [])}
    out['ALL'] = data.get('overall', {})
    return out

base = load_metrics(BASE_METRICS)
v1 = load_metrics(V1_METRICS)
v2 = load_metrics(METRICS_JSON)
scenarios = ['ALL', 'clean', 'noise', 'reverb', 'dropout', 'far_field']
rows_out = []
for scenario in scenarios:
    b = float(base.get(scenario, {}).get('error_rate', 0.0) or 0.0)
    one = float(v1.get(scenario, {}).get('error_rate', 0.0) or 0.0)
    two = float(v2.get(scenario, {}).get('error_rate', 0.0) or 0.0)
    rows_out.append({
        'scenario': scenario,
        'base_wer': b,
        'v1_wer': one,
        'v2_wer': two,
        'v2_minus_base': two - b,
        'v2_relative_delta': ((two - b) / b) if b else None,
        'v2_minus_v1': two - one,
    })

df = pd.DataFrame(rows_out)
display(df)

noise_delta = df[df['scenario'] == 'noise'].iloc[0]['v2_minus_base']
reverb_delta = df[df['scenario'] == 'reverb'].iloc[0]['v2_minus_base']
clean_rel = df[df['scenario'] == 'clean'].iloc[0]['v2_relative_delta']
print('noise v2_minus_base =', noise_delta)
print('reverb v2_minus_base =', reverb_delta)
print('clean relative delta =', clean_rel)
if noise_delta < 0 or reverb_delta < 0:
    print('v2 有希望：noise 或 reverb 至少一个场景优于 base。下一步再扩大 step 或做 clean regression/router 分析。')
else:
    print('v2 仍未赢 base：继续做 target/data/lr ablation，不进入 router。')


In [ ]:
# 列出 v2 产物，后续可用 00_github_commit_push_colab.ipynb 提交受控输出。
for root in [OUTPUT_DIR, EVAL_DIR]:
    print('\n===', root, '===')
    for path in sorted(root.rglob('*')):
        if path.is_file():
            print(path.relative_to(PROJECT_DIR), path.stat().st_size)
